## lastfm-artist-weights.ipynb
Attaches a Last.fm popularity **weight** (total play count) to each MusicBrainz artist.

Uses the [HetRec Last.fm dataset](readme.txt) in this folder:
- `artists.dat`      — Last.fm artist id + name
- `user_artists.dat` — per-user listen weight per Last.fm artist id

Last.fm has no MusicBrainz IDs, so artists are matched by normalised name.

**Input:** `../data/pickles/final_artist_df.pkl` (produced by `1-EDA/02-parquet-to-dataframes.ipynb`).
**Output:** the same pickle, augmented with `lastfm_id` and `weight` columns.

Run after `1-EDA/02`.

In [ ]:
import pandas as pd

PICKLE_PATH = '../data/pickles/final_artist_df.pkl'
ARTISTS_DAT = 'artists.dat'        # Last.fm artist id + name (this folder)
WEIGHTS_DAT = 'user_artists.dat'   # per-user listen weight per artist id
ARTIST_NAME_COL = 'name'           # MusicBrainz artist name column in final_artist_df

In [ ]:
# MusicBrainz artist table (built by 1-EDA/02)
final_artist_df = pd.read_pickle(PICKLE_PATH)

# Last.fm artists + total listen weight per Last.fm artist id
lastfm_artists = pd.read_csv(ARTISTS_DAT, sep='\t', usecols=['id', 'name'])
user_artists   = pd.read_csv(WEIGHTS_DAT, sep='\t', usecols=['artistID', 'weight'])
weights_grouped = user_artists.groupby('artistID')['weight'].sum().reset_index()

lastfm_data = (
    lastfm_artists
    .merge(weights_grouped, left_on='id', right_on='artistID', how='left')
    .drop(columns=['artistID'])
    .rename(columns={'id': 'lastfm_id', 'name': 'lastfm_name'})
)

print(f'MusicBrainz artists : {len(final_artist_df):,}')
print(f'Last.fm artists     : {len(lastfm_data):,}')

In [ ]:
# Match Last.fm -> MusicBrainz by normalised name (no shared IDs exist)
def normalize(s):
    return '' if pd.isna(s) else str(s).strip().lower()

final_artist_df['_name_norm'] = final_artist_df[ARTIST_NAME_COL].apply(normalize)
lastfm_data['_name_norm']     = lastfm_data['lastfm_name'].apply(normalize)

# Drop any prior attempt so a re-run is idempotent
final_artist_df = final_artist_df.drop(columns=[c for c in ['lastfm_id', 'weight'] if c in final_artist_df.columns])

final_artist_df = final_artist_df.merge(
    lastfm_data[['lastfm_id', 'weight', '_name_norm']],
    on='_name_norm', how='left',
).drop(columns=['_name_norm'])

final_artist_df['weight'] = final_artist_df['weight'].fillna(0)
final_artist_df['lastfm_id'] = pd.array(final_artist_df['lastfm_id'], dtype=pd.Int64Dtype())
final_artist_df = final_artist_df.drop_duplicates(subset=['id'], keep='first')

total   = len(final_artist_df)
matched = final_artist_df['lastfm_id'].notna().sum()
print(f'Total artists        : {total:,}')
print(f'Matched with Last.fm : {matched:,} ({matched/total*100:.1f}%)')
print(f'Max weight           : {final_artist_df["weight"].max():,.0f}')
final_artist_df.sort_values('weight', ascending=False)[[ARTIST_NAME_COL, 'lastfm_id', 'weight']].head(10)

In [ ]:
final_artist_df.to_pickle(PICKLE_PATH)
print(f'Saved augmented artist table -> {PICKLE_PATH}')